In [1]:
from torch.utils.data import random_split
import torchvision.models.segmentation as segmentation
import torch.nn as nn
import torch.optim as optim
import numpy as np

import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet

try: # disable certificate verification, needed on MacOS
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context
except ImportError:
    pass  # SSL module not available, skipping workaround

In [2]:
class SegmentationTransform:
    def __init__(self, size=(256, 256)):
        self.size = size
        self.image_transform = transforms.Compose([
            transforms.Resize(self.size),
            transforms.ToTensor()
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize(self.size, interpolation=transforms.InterpolationMode.NEAREST),
            transforms.PILToTensor()
        ])

    def __call__(self, image, target):
        image = self.image_transform(image)
        target = self.mask_transform(target).squeeze(0).long() - 1  # [1, H, W] → [H, W], label remap
        return image, target

train_dataset = OxfordIIITPet(
    root='.',
    target_types='segmentation',
    download=True,
    transforms=SegmentationTransform(size=(256, 256))
)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

def load_images(filename: str, image_shape: tuple = (3, 256, 256)):
    df = pd.read_csv(filename)
    images = []
    for _, row in df.iterrows():
        flat = eval(row['image'])  # Convert stringified list back to list
        tensor = torch.tensor(flat, dtype=torch.float32).reshape(image_shape)
        images.append(tensor)
    return torch.stack(images)  # [B, C, H, W]

def compute_iou(mask_true: np.ndarray, mask_pred: np.ndarray, num_classes: int = 3) -> float:
    """Computes mean Intersection over Union (mIoU) for multi-class masks."""
    ious = []
    for cls in range(num_classes):
        true_cls = mask_true == cls
        pred_cls = mask_pred == cls
        intersection = np.logical_and(true_cls, pred_cls).sum()
        union = np.logical_or(true_cls, pred_cls).sum()
        if union == 0:
            continue  # skip this class — not present in either pred or true
        iou = intersection / union
        ious.append(iou)
    if not ious:
        return 1.0  # If no classes are present in either mask, define mIoU as perfect
    return np.mean(ious)

## Split

In [3]:
# Dataset and split
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create loaders
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

## Model

In [4]:
class SegmentationModel(nn.Module):
    def __init__(self, num_classes=3, freeze=True):
        super().__init__()
        # Load the lightweight MobileNetV3 DeepLab model (because DeepLabV3 slow af - 2 epochs after 3 hours, this one took only 46 minutes)
        weights = segmentation.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
        self.model = segmentation.deeplabv3_mobilenet_v3_large(weights=weights)

        # Freeze the backbone based on the parameter, as seen in NN_finetuning.ipynb
        if freeze:
            for param in self.model.backbone.parameters():
                param.requires_grad = False

        # Replace the classifier head for 3 classes (background, pet, edge)
        in_channels = self.model.classifier[4].in_channels
        self.model.classifier[4] = nn.Conv2d(in_channels, num_classes, kernel_size=1, stride=1)

        # Handle auxiliary classifier if present
        if self.model.aux_classifier is not None:
            in_channels_aux = self.model.aux_classifier[4].in_channels
            self.model.aux_classifier[4] = nn.Conv2d(in_channels_aux, num_classes, kernel_size=1, stride=1)

    def forward(self, x):
        # torchvision segmentation models output a dictionary. We want the main output.
        return self.model(x)['out']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SegmentationModel(freeze=True).to(device)

In [5]:
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
    print(f"Train Loss: {avg_loss:.4f}")
    return total_loss / len(dataloader)
#use the same code as in practicals but keep the result


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    val_ious = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()

            # Calculate metrics
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            masks_np = targets.cpu().numpy()
            for i in range(preds.shape[0]):
                val_ious.append(compute_iou(masks_np[i], preds[i]))

    avg_loss = total_loss / len(dataloader)
    avg_iou = np.mean(val_ious)
    return avg_loss, avg_iou
#adapt code from practicals for semantic segmentation - classifying every single pixel in the image

## Phase 1 — Head only, backbone frozen


- `ReduceLROnPlateau` — halves LR after 2 epochs without val mIoU improvement (fixes the stall you saw at epoch 6)
- Early stopping with patience=4 — stops wasting epochs once we've plateaued

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)

epochs           = 15
best_iou         = 0.0
patience         = 4
patience_counter = 0

for epoch in range(epochs):
    train_loss = train(model, train_loader, criterion, optimizer, device)
    val_loss, val_iou = validate(model, val_loader, criterion, device)
    scheduler.step(val_iou)

    print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val mIoU: {val_iou:.4f}")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), 'best_model.pth')
        print("Best model saved!")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

print(f"Phase 1 best val mIoU: {best_iou:.4f}")

Train Loss: 0.3161
Epoch 1/15 - Train Loss: 0.3161, Val Loss: 0.2775, Val mIoU: 0.6913
Best model saved!
Train Loss: 0.2581
Epoch 2/15 - Train Loss: 0.2581, Val Loss: 0.2689, Val mIoU: 0.7017
Best model saved!
Train Loss: 0.2426
Epoch 3/15 - Train Loss: 0.2426, Val Loss: 0.2568, Val mIoU: 0.7005
Train Loss: 0.2280
Epoch 4/15 - Train Loss: 0.2280, Val Loss: 0.2503, Val mIoU: 0.7135
Best model saved!
Train Loss: 0.2176
Epoch 5/15 - Train Loss: 0.2176, Val Loss: 0.2896, Val mIoU: 0.6899
Train Loss: 0.2087
Epoch 6/15 - Train Loss: 0.2087, Val Loss: 0.2555, Val mIoU: 0.7152
Best model saved!
Train Loss: 0.2042
Epoch 7/15 - Train Loss: 0.2042, Val Loss: 0.2565, Val mIoU: 0.7152
Best model saved!
Train Loss: 0.1957
Epoch 8/15 - Train Loss: 0.1957, Val Loss: 0.2594, Val mIoU: 0.7149
Train Loss: 0.1876
Epoch 9/15 - Train Loss: 0.1876, Val Loss: 0.2683, Val mIoU: 0.7130
Train Loss: 0.1738
Epoch 10/15 - Train Loss: 0.1738, Val Loss: 0.2664, Val mIoU: 0.7167
Best model saved!
Train Loss: 0.1673
Ep

## Phase 2 — Unfreeze backbone, fine-tune end-to-end

This is where the real gain comes from. The frozen backbone was limiting what the model could learn about pets. We load the best phase-1 weights and continue at 10x lower LR so we don't destroy the pretrained features.

In [ ]:
# Load best weights from phase 1
model.load_state_dict(torch.load('best_model.pth'))

# Unfreeze backbone
for param in model.model.backbone.parameters():
    param.requires_grad = True

# 10x lower LR — fine-tuning pretrained weights needs a gentler touch
optimizer_ft = optim.Adam(model.parameters(), lr=1e-4)
scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(optimizer_ft, mode='max', patience=2, factor=0.5)

epochs_ft        = 10
patience_counter = 0

for epoch in range(epochs_ft):
    train_loss = train(model, train_loader, criterion, optimizer_ft, device)
    val_loss, val_iou = validate(model, val_loader, criterion, device)
    scheduler_ft.step(val_iou)

    print(f"[Fine-tune] Epoch {epoch + 1}/{epochs_ft} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val mIoU: {val_iou:.4f}")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), 'best_model.pth')
        print("Best model saved!")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

print(f"Final best val mIoU: {best_iou:.4f}")

## Generate predictions and save submission

In [ ]:
test_path = r"C:\Users\Richard\Desktop\daša škola\4. ročník\data science 2\ukol2\test_images.csv"
df = pd.read_csv(test_path)
ids = df.iloc[:, 0].values

# 1. Load test data and best model
test_tensor = load_images(test_path).to(device)
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# 2. Predict
predictions = []
with torch.no_grad():
    outputs = model(test_tensor)
    # Shape becomes [Batch, 256, 256]
    preds = torch.argmax(outputs, dim=1).cpu().numpy()

    # 3. Flatten and stringify
    for i in range(preds.shape[0]):
        flat_mask = preds[i].flatten().tolist()
        predictions.append(str(flat_mask))

# 4. Save to CSV
submission_df = pd.DataFrame({
    'id': ids,
    'mask': predictions,
    'Usage': 'Public'
})
submission_df.to_csv('submission3.csv', index=False)
#print('Submission saved!')